# 03 — GRPO Training: Safe Medical Triage

**Objective:** Train Qwen3-8B with GRPO to improve tool-using policy over the prompt-only baseline.

**Prerequisites:**
- `01_env_smoke_and_oracle.ipynb` — environment and oracle verified
- `02_baseline_vllm_eval.ipynb` — baseline metrics collected

**Hardware:** RTX 6000 Pro Blackwell (96 GB VRAM). Training in fp16 without quantization — full precision LoRA.

## 1. Install dependencies

In [ ]:
%%capture
!pip install -q "unsloth>=2025.2" "trl>=0.22" "transformers>=4.50" \
    "accelerate>=1.0" "bitsandbytes>=0.44" "peft>=0.12" \
    "datasets>=2.20" "sentencepiece>=0.2" "tqdm>=4.66" \
    "pandas>=2.2" "numpy>=1.26"
print("Dependencies installed.")

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
if not (PROJECT_ROOT / "triage").is_dir():
    for candidate in [Path("/kaggle/working"), Path("/content"), Path.home()]:
        if (candidate / "triage").is_dir():
            PROJECT_ROOT = candidate
            break

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))
!pip install -q -e .

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"triage/ exists: {(PROJECT_ROOT / 'triage').is_dir()}")

## 2. Generate train/eval datasets

In [ ]:
from triage import generate_train_val_datasets, generate_eval_datasets, run_oracle_on_dataset

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_val = generate_train_val_datasets(
    out_dir=DATA_DIR,
    train_size=5000,
    val_size=500,
    seed=13,
    difficulties=range(1, 11),
    progress=True,
)
print("Train/val:")
display(train_val.summary_df())

eval_ds = generate_eval_datasets(
    out_dir=DATA_DIR / "eval",
    size_per_difficulty=100,
    seed=101,
    difficulties=range(1, 11),
    progress=True,
)
print("\nEval buckets:")
display(eval_ds.summary_df())

In [ ]:
# Oracle sanity check
oracle_d1 = run_oracle_on_dataset(
    eval_ds.paths["eval_d1"],
    out_metrics=DATA_DIR / "eval" / "oracle_d1.metrics.jsonl",
    progress=True,
)
print(f"Oracle eval_d1: success={oracle_d1.summary['success_rate']:.0%}  "
      f"reward={oracle_d1.summary['avg_total_reward']:.2f}")
assert oracle_d1.summary["success_rate"] == 1.0, "Oracle must be 100%"

## 3. Configure GRPO training

**RTX 6000 Pro Blackwell (96 GB VRAM) config:**

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `load_in_4bit` | `False` | fp16 — full precision, 96 GB handles it easily |
| `lora_r` | `64` | higher rank = more capacity (4-bit would need r=32) |
| `per_device_train_batch_size` | `4` | large batch fits in VRAM |
| `gradient_accumulation_steps` | `2` | effective batch = 8 |
| `num_generations` | `8` | better GRPO variance reduction |
| `enable_thinking` | `False` | matches baseline eval — direct action generation |

In [ ]:
from runtimes.train_unsloth import UnslothGRPOConfig

TRAIN_MODEL = "Qwen/Qwen3-8B"
RUN_NAME = "qwen3_8b_grpo_fp16"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "runs" / RUN_NAME
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

cfg = UnslothGRPOConfig(
    dataset=str(DATA_DIR / "train.jsonl"),
    model=TRAIN_MODEL,
    output_dir=str(ARTIFACTS_DIR),

    # ── Sequence lengths ──
    max_seq_length=4096,
    max_prompt_length=2048,
    max_completion_length=1536,

    # ── RTX 6000 Pro: fp16, no quantization ──
    enable_thinking=False,
    load_in_4bit=False,
    lora_r=64,
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,  # effective batch = 8
    num_train_epochs=1.0,
    num_generations=8,              # 8 completions per prompt
    save_steps=100,
    logging_steps=1,
    warmup_steps=10,
    seed=42,

    trust_remote_code=True,
    use_unsloth_gradient_checkpointing=True,
    report_to="none",
    unsloth_disable_compile=True,
    clear_unsloth_cache=True,
)

cfg.save_json(ARTIFACTS_DIR / "train_config.json")

print("Training config:")
for k, v in cfg.to_dict().items():
    if k.startswith("unsloth_") or k.startswith("disable_") or k.startswith("clear_"):
        continue
    print(f"  {k}: {v}")

## 4. Prepare GRPO session

In [ ]:
from runtimes.train_unsloth import prepare_grpo_session

session = prepare_grpo_session(
    cfg,
    progress=True,
    max_cases=None,   # set to e.g. 200 for a quick debug run
)

print(f"Model: {cfg.model}  (fp16, no quantization)")
print(f"Training cases: {len(session.cases)}")
print(f"LoRA rank: {cfg.lora_r}")
print(f"Trainable params: {sum(p.numel() for p in session.model.parameters() if p.requires_grad):,}")
print(f"Total params: {sum(p.numel() for p in session.model.parameters()):,}")

## 5. Inspect a training prompt

In [ ]:
row = session.train_rows[0]
messages = row["prompt"]

print(f"Case: {row['case_id']}  Difficulty: {row['difficulty']}")
print(f"Num messages: {len(messages)}")
print()
print("=== SYSTEM ===")
print(messages[0]["content"])
print()
print("=== USER (last 500 chars) ===")
print(messages[1]["content"][-500:])

## 6. Run GRPO training

With RTX 6000 Pro (96 GB): expect ~30-60 min for 5000 cases × 1 epoch.
Monitor `reward/mean` — should increase from ~-1.6 toward 0.

In [ ]:
import time

print(f"Starting GRPO training on {cfg.model}...")
print(f"Effective batch: {cfg.per_device_train_batch_size} × {cfg.gradient_accumulation_steps} = {cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps}")
print(f"Generations per prompt: {cfg.num_generations}")
print()

t0 = time.time()
session.trainer.train()
elapsed = time.time() - t0

print(f"\nTraining completed in {elapsed/60:.1f} min ({elapsed/3600:.2f} h)")

## 7. Save adapter & export merged model

In [ ]:
from runtimes.train_unsloth import finalize_training_artifacts, export_merged_model_for_vllm

# Save adapter
result = finalize_training_artifacts(session)
print(f"Adapter: {result.adapter_dir}")
print(f"Manifest: {result.manifest_path}")

# Export merged 16-bit for vLLM
export_result = export_merged_model_for_vllm(
    result.adapter_dir,
    base_model=TRAIN_MODEL,
    run_name=RUN_NAME,
    export_dir=ARTIFACTS_DIR / "merged_16bit",
    manifest=result.manifest_path,
    max_seq_length=cfg.max_seq_length,
)

MERGED_MODEL_PATH = str(export_result["export_dir"])
MANIFEST_PATH = str(export_result["manifest_path"])
print(f"\nMerged model: {MERGED_MODEL_PATH}")
print(f"Manifest: {MANIFEST_PATH}")

## 8. Evaluate: trained model vs baseline

Run both models through interactive rollout on the same eval sets.

In [ ]:
try:
    import vllm
    print(f"vLLM {vllm.__version__}")
except ImportError:
    print("Installing vLLM...")
    !pip install -q "vllm>=0.10.2" "openai>=1.40"
    import vllm
    print(f"vLLM {vllm.__version__} installed")

In [ ]:
from runtimes.eval_vllm import VLLMEvalConfig, run_vllm_rollouts

def make_eval_cfg(dataset_path, model=None, manifest=None, tag="eval"):
    return VLLMEvalConfig(
        dataset=str(dataset_path),
        model=model,
        manifest=manifest,
        out_traj=str(ARTIFACTS_DIR / f"{tag}.traj.jsonl"),
        out_metrics=str(ARTIFACTS_DIR / f"{tag}.metrics.jsonl"),
        max_model_len=4096,
        max_tokens=2048,
        temperature=0.0,
        top_p=1.0,
        rollout_mode="interactive",
        enable_thinking=False,
        tensor_parallel_size=1,
        trust_remote_code=True,
    )

In [ ]:
# ── eval_d1: quick sanity ──
print("=== eval_d1 ===")

cfg_t = make_eval_cfg(eval_ds.paths["eval_d1"], manifest=MANIFEST_PATH, tag="trained_d1")
trained_d1 = run_vllm_rollouts(cfg_t)

cfg_b = make_eval_cfg(eval_ds.paths["eval_d1"], model=TRAIN_MODEL, tag="baseline_d1")
baseline_d1 = run_vllm_rollouts(cfg_b)

print(f"Baseline: success={baseline_d1.summary['success_rate']:.0%}  reward={baseline_d1.summary['avg_total_reward']:.2f}")
print(f"GRPO:     success={trained_d1.summary['success_rate']:.0%}  reward={trained_d1.summary['avg_total_reward']:.2f}")

## 9. Full comparison: d1..d10

In [ ]:
import pandas as pd

results = {}
for d in range(1, 11):
    key = f"d{d}"
    ds_path = eval_ds.paths[f"eval_{key}"]

    cfg_t = make_eval_cfg(ds_path, manifest=MANIFEST_PATH, tag=f"trained_{key}")
    results[f"GRPO_{key}"] = run_vllm_rollouts(cfg_t)

    cfg_b = make_eval_cfg(ds_path, model=TRAIN_MODEL, tag=f"baseline_{key}")
    results[f"Baseline_{key}"] = run_vllm_rollouts(cfg_b)

    print(f"d{d}: baseline={results[f'Baseline_{key}'].summary['success_rate']:.0%}  "
          f"grpo={results[f'GRPO_{key}'].summary['success_rate']:.0%}")

In [ ]:
# ── Comparison table ──
rows = []
for d in range(1, 11):
    for label in ["Baseline", "GRPO"]:
        r = results[f"{label}_d{d}"]
        rows.append({
            "model": label, "difficulty": d,
            "success_rate": r.summary.get("success_rate"),
            "avg_reward": r.summary.get("avg_total_reward"),
            "avg_tool_calls": r.summary.get("avg_tool_calls"),
            "avg_ev_coverage": r.summary.get("avg_evidence_coverage"),
            "avg_violations": r.summary.get("avg_policy_violations"),
            "duplicate_q_rate": r.summary.get("duplicate_question_rate"),
        })

df_all = pd.DataFrame(rows)

print("\n=== Success Rate ===")
display(df_all.pivot_table(index="difficulty", columns="model", values="success_rate").round(2))

print("\n=== Average Reward ===")
display(df_all.pivot_table(index="difficulty", columns="model", values="avg_reward").round(2))

print("\n=== Evidence Coverage ===")
display(df_all.pivot_table(index="difficulty", columns="model", values="avg_ev_coverage").round(2))

print("\n=== Duplicate Question Rate ===")
display(df_all.pivot_table(index="difficulty", columns="model", values="duplicate_q_rate").round(3))

In [ ]:
# ── Failure reasons ──
print("Failure reasons by difficulty:")
for d in range(1, 11):
    b_fr = results[f"Baseline_d{d}"].summary.get("failure_reasons", {})
    g_fr = results[f"GRPO_d{d}"].summary.get("failure_reasons", {})
    print(f"  d{d}: Baseline={b_fr}  GRPO={g_fr}")

## 10. Analysis & discussion

### Baseline failure patterns (from d1 eval)
| Issue | Count | Root Cause |
|-------|-------|------------|
| duplicate_questions | 9 in 4/10 cases | Model repeats Q_RASH_SPREAD 3-4× |
| irrelevant_questions | 8 in 5/10 cases | Asks Q_LOCALIZATION where not needed |
| wrong_final_decision | 3/10 | Always picks BOOK_SAME_DAY |
| premature_finish | 4/10 | Finishes with only 50% evidence |
| Never calls `lookup_protocol` | 10/10 | Doesn't use protocol navigation |

### What GRPO should improve
- **duplicate/irrelevant questions** → shaping penalties -0.05/-0.04 per occurrence
- **evidence coverage** → success requires 100% coverage + correct disposition
- **disposition diversity** → success reward +1.0 only for correct disposition
- **protocol usage** → cases need protocol info for correct triage

### Proxy reward risks
1. **Shortcutting**: model may learn to always pick SELF_CARE (no booking/escalation needed → avoids confirmation mistakes)
2. **Minimal trajectory**: heavy step/tool penalties may discourage exploration
3. **Difficulty imbalance**: if d1 cases are overrepresented, model may overfit to simple patterns

### Mitigation ideas
- Balance training data across difficulties
- Reduce step_penalty for harder cases (allow longer trajectories)
- Add explicit bonus for evidence_coverage (currently only implicit through success)